In [1]:
from pathlib import Path
import pandas as pd

# データ読み込み
DATA_PATH = Path("../../data/raw/assistments_2009_2010/skill_builder_data.csv")

# 対象スキル（ここだけ変える）
# TARGET_SKILLS = [280, 277, 47, 70, 85, 58, 311, 49, 18, 74, 279, 50, 13, 67, 77, 15, 79, 278]
# TARGET_SKILLS = [47, 49, 277, 279, 280] # 候補1/9
# TARGET_SKILLS = [47, 49, 58, 70, 74, 79, 277, 278, 279, 280]
TARGET_SKILLS = [10, 13, 14, 15, 47, 49, 58, 70, 74, 77, 79, 277, 278, 279, 280]

print(' '.join(map(str, TARGET_SKILLS)))

# 67, 86, 309

10 13 14 15 47 49 58 70 74 77 79 277 278 279 280


In [2]:
df = pd.read_csv(DATA_PATH, encoding="latin1")

df = df[
    ["user_id", "order_id", "template_id", "skill_id", "skill_name"]
].dropna()

for c in ["user_id", "order_id", "template_id", "skill_id"]:
    df[c] = df[c].astype(int)

# 対象スキルのみ
df_skill = df[df["skill_id"].isin(TARGET_SKILLS)].copy()
df_skill = df_skill.sort_values(["user_id", "order_id"])

/var/folders/zg/773ptkr55z99zw26dvy19_v00000gn/T/ipykernel_63687/579294825.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, encoding="latin1")


In [3]:
def first_half_skills(user_df):
    n = len(user_df)
    first = user_df.iloc[: n // 2]
    return set(first["skill_id"].unique())

target_set = set(TARGET_SKILLS)

user_first_skills = (
    df_skill
    .groupby("user_id")
    .apply(first_half_skills)
)

valid_users = user_first_skills[
    user_first_skills.apply(lambda s: target_set.issubset(s))
].index

n_users = len(valid_users)

print(f"Valid users (first half covers all skills): {n_users}")


Valid users (first half covers all skills): 94


/var/folders/zg/773ptkr55z99zw26dvy19_v00000gn/T/ipykernel_63687/1938312698.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(first_half_skills)


In [4]:
df_valid = df_skill[df_skill["user_id"].isin(valid_users)]

n_templates = df_valid["template_id"].nunique()

print(f"Templates used by valid users: {n_templates}")


Templates used by valid users: 139


In [5]:
print("=== SUMMARY ===")
print(f"Skills: {TARGET_SKILLS}")
print(f"Users: {n_users}")
print(f"Templates: {n_templates}")


=== SUMMARY ===
Skills: [10, 13, 14, 15, 47, 49, 58, 70, 74, 77, 79, 277, 278, 279, 280]
Users: 94
Templates: 139


In [6]:
# TARGET_SKILLS をすべて解いているユーザのうち、その skill_id も解いている人数

TOP_N = 10                                # 上位何件見るか

K = len(set(TARGET_SKILLS))
target_set = set(TARGET_SKILLS)

# スキル名のマッピングを取得
skill_id_to_name = df[['skill_id', 'skill_name']].drop_duplicates().set_index('skill_id')['skill_name'].to_dict()

# df ではなく df_skill (対象スキルのみ) または df_valid を使う
# ここでは全体から探すため、dfを使うが、必要な列だけに絞る
print("Processing user-skill pairs...")

# 必要な列だけ抽出してメモリ節約
user_skill = df[["user_id", "skill_id"]].drop_duplicates(["user_id", "skill_id"])
print(f"Unique (user, skill) pairs: {len(user_skill):,}")

# 1) TARGET_SKILLS をすべて含むユーザ集合(全期間)
print("Finding users with all target skills...")
target_hits = user_skill[user_skill["skill_id"].isin(target_set)]
users_with_all_targets = (
    target_hits.groupby("user_id")["skill_id"]
    .nunique()
    .pipe(lambda s: s[s == K].index)  # loc[lambda] より効率的
)

n_users_all_targets = len(users_with_all_targets)
print(f"Users who attempted ALL TARGET_SKILLS (anytime): {n_users_all_targets:,}")

# 2) そのユーザ群におけるスキル共起(= そのスキルにも触れているユーザ数)
print("Calculating co-occurrence...")
cooccur = (
    user_skill[user_skill["user_id"].isin(users_with_all_targets)]
    .groupby("skill_id")["user_id"]
    .nunique()
    .sort_values(ascending=False)
)

# TARGET_SKILLS 自身は除外
cooccur = cooccur.drop(labels=list(target_set), errors="ignore")

# 3) ランキング表(追加したとき残るユーザ数の目安も同じ)
res = cooccur.head(TOP_N).reset_index()
res.columns = ["skill_id", "n_users_cooccur"]

# スキル名を追加
res["skill_name"] = res["skill_id"].map(lambda x: skill_id_to_name.get(x, f'Unknown ({x})'))

res["cooccur_rate"] = res["n_users_cooccur"] / n_users_all_targets  # P(skill | all targets)
res["n_users_if_added"] = res["n_users_cooccur"]                    # 追加しても残る最大ユーザ数(全期間定義)

# 列の順序を調整
res = res[["skill_id", "skill_name", "n_users_cooccur", "cooccur_rate", "n_users_if_added"]]

print("\n=== Top co-occurring skills ===")
display(res)

Processing user-skill pairs...
Unique (user, skill) pairs: 39,382
Finding users with all target skills...
Users who attempted ALL TARGET_SKILLS (anytime): 190
Calculating co-occurrence...

=== Top co-occurring skills ===


,skill_id,skill_name,n_users_cooccur,cooccur_rate,n_users_if_added
0,67,Multiplication Fractions,190,1.000000,190
1,81,Unit Rate,190,1.000000,190
2,50,Ordering Fractions,189,0.994737,189
3,86,Exponents,189,0.994737,189
4,309,"Order of Operations +,-,/,* () positive reals",189,0.994737,189
5,18,Probability of a Single Event,187,0.984211,187
6,311,Equation Solving Two or Fewer Steps,185,0.973684,185
7,61,Division Fractions,185,0.973684,185
8,34,Unit Conversion Within a System,185,0.973684,185
9,1,Box and Whisker,182,0.957895,182
